In [ ]:
import os
from google.colab import userdata,drive
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN']= userdata.get('KAGGLE_KEY')
!kaggle datasets download -d syamkakarla/pavia-university-hsi
print("the dataset has been downloaded")


In [ ]:
!unzip -nq pavia-university-hsi.zip -d /content/PaviaU
print(" the folder has been unzipped")

In [ ]:
import scipy.io as sio
data_dict= sio.loadmat('/content/PaviaU/PaviaU.mat')
gt_dict= sio.loadmat('/content/PaviaU/PaviaU_gt.mat')
print("ground truth keys:", [k for k in gt_dict.keys() if not k.startswith('__')])
print("data keys:", [k for k in data_dict.keys() if not k.startswith('__')])

In [ ]:
#extracting the numpy arrays from the dicts
X= data_dict['paviaU']
y= gt_dict['paviaU_gt']
#analysing the size of the arrays
print("Input data shape:", X.shape)
print("ground truth shape:", y.shape)

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import layers,models
import time
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

In [ ]:
# I just dont like the labels as integers, creating a dictionary to map [integer labels] to the [name of the material]
class_map={
    0: "Asphalt",
    1: "Meadows",
    2: "Gravel",
    3: "trees",
    4: "Painted Metal Sheets",
    5: "Bare Soil",
    6: "Bitumen",
    7: "Self-Blocking Bricks",
    8: "Shadows"
}

In [ ]:
total_samples= np.sum(y>0)
print("total valid samples:",total_samples)
ignored_samples= np.sum(y==0)
print("samples with 0 label:", ignored_samples)
def class_count(class_num):
  class_count= np.sum(y==class_num+1)
  return class_count
for i in range(0,9):
  print(class_map[i],": ",class_count(i))

In [ ]:
#creating patches out of image
def create_patches(X,y,patch_size=7):
  margin= patch_size//2
  X_padded= np.pad(X, ((margin,margin),(margin,margin),(0,0)), mode='constant')
  all_patches = np.lib.stride_tricks.sliding_window_view(X_padded, (patch_size, patch_size), axis=(0, 1))
  row_indices, col_indices = np.where(y > 0)
  X_patches=all_patches[row_indices,col_indices]
  X_patches= np.moveaxis(X_patches,1,-1)
  X_patches= np.expand_dims(X_patches,-1)
  y_gt= y[row_indices,col_indices]-1
  return X_patches, y_gt

In [ ]:
X_patches, y_gt = create_patches(X,y)
#splitting the data into training, validation and testing set
X_temp, X_test, y_temp, y_test= train_test_split(X_patches,y_gt, test_size=0.1,random_state=42,stratify=y_gt)
X_train,X_val,y_train,y_val= train_test_split(X_temp,y_temp, train_size=0.8889, random_state=42, stratify=y_temp)
#converting numpy arrays into tensorflow tensors for better processing
X_train,X_test,X_val= [tf.convert_to_tensor(X, dtype=tf.float32) for X in (X_train,X_test,X_val)]
y_train,y_test,y_val=[tf.convert_to_tensor(y,dtype=tf.int32) for y in (y_train,y_test,y_val)]
print("X_train:",X_train.shape,"y_train:",y_train.shape,"  X_test:",X_test.shape,"y_test:",y_test.shape,"  X_val:",X_val.shape,"y_val:",y_val.shape)

In [ ]:
# storing the train, test and validation split as numpy arrays in my google drive
import shutil
np.savez_compressed(
    'pavia_data_split.npz',
    X_train= X_train.numpy(),
    X_val= X_val.numpy(),
    X_test=X_test.numpy(),
    y_train=y_train.numpy(),
    y_val=y_val.numpy(),
    y_test=y_test.numpy()

)

target_drive_path= '/content/drive/MyDrive/trained_models'
shutil.copy('pavia_data_split.npz', target_drive_path)
print("the data split has been stored into the google drive as a compressed .npz file")

In [ ]:
# normalizing the dataset using Z-score
mean= tf.reduce_mean(X_train, axis=(0,1,2), keepdims=True)
std= tf.math.reduce_std(X_train, axis=(0,1,2), keepdims=True)
epsilon= 1e-8
X_train= (X_train-mean)/std+epsilon
X_test= (X_test-mean)/std+epsilon
X_val= (X_val-mean)/std+epsilon

In [ ]:
#visualizing the HSI image
def visualization(X_train,y_train,patch_num):
  patch= tf.squeeze(X_train[patch_num])
  label= y_train[patch_num]
  patch= patch.numpy()
  label=label.numpy()
  fig, axes= plt.subplots(1,3,figsize=(15,5))

  axes[0].imshow(patch[:,:,1],cmap='gray')
  axes[0].set_title(f"band 1: {class_map[label]}")

  axes[1].imshow(patch[:,:,102],cmap='gray')
  axes[1].set_title(f"band 102: {class_map[label]}")

  central_pixel_spectrum=patch[2,2,:]
  axes[2].plot(central_pixel_spectrum, color='crimson', linewidth=3)
  axes[2].set_xlabel("spectral band")
  axes[2].set_ylabel("reflectance intensity")
  axes[2].set_title("spectral curve")

  plt.tight_layout()
  plt.show()

visualization(X_train,y_train,56)

In [ ]:
# using EarlyStopping in order to stop the training when maximum validation accuray is obtained in an epoch
# using ModelCheckpoint to save the model after each epoch into my drive
# Both the EarlyStopping and ModelCheckpoint are callbacks bundled into a list callbacks=[callback1, callback2]
callbacks_list=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
           ModelCheckpoint(filepath='best_performing_model.keras', monitor='val_loss', save_best_only=True)]


In [ ]:
# The model will have 2 3D convulation layers and 3 Dense layers, we are not using maxpool since it will reduce the data too much
model= models.Sequential([
    layers.Conv3D(16, (3,3,11), activation='relu', padding='valid', input_shape=(7,7,103,1)),
    layers.Conv3D(32,(3,3,11), activation='relu',padding='valid'),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.35),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(9,activation='softmax')
])
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'] )
training= model.fit(X_train,y_train, batch_size=64, epochs=50, validation_data=(X_val,y_val), verbose=1, callbacks=callbacks_list)


In [ ]:
os.makedirs('/content/drive/MyDrive/trained_models', exist_ok=True)
model.save('/content/drive/MyDrive/trained_models/HSI_CNN.keras')

In [ ]:
# Since we will be comparing the model, I want to use all the metrics under a single class
class model_performace_metrics:
  def __init__(self,model,model_file_path,class_map):
    self.model= model
    self.model_file_path= model_file_path
    self.class_map= class_map

  def scorecard(self, X_test,y_test):

    #calculating total number of parameters in the model
    total_params= self.model.count_params()

    #calculating the disc space occupied by the model
    model_size= os.path.getsize(self.model_file_path)/(1024*1024)

    #calculating overall prediction accuracy
    prediction= self.model.evaluate(X_test,y_test)

    #calculating the inference latency (time delay for model to make a prediction)
    warmup_run= model.predict(X_test[:10],verbose=0)
    start_time= time.time()
    actual_run= model.predict(X_test,verbose=0)
    end_time= time.time()
    inference_latency= (end_time-start_time)*1000
    per_patch_latency= inference_latency/X_test.shape[0]

    #calculating precision per class
    y_pred= np.argmax(actual_run, axis=1)
    precision_per_class= precision_score(y_test, y_pred, average=None)*100

    #calculating recall per class
    recall_per_class= recall_score(y_test, y_pred, average=None)*100

    #printing the layout of the scorecard
    print("Edge AI deployment parameters")
    print("total parameters:",total_params)
    print(f"Disc Storage size: {model_size:.2f} MB")
    print(f"inference latency: {inference_latency:.2f} ms")
    print(f"inference latency per patch: {per_patch_latency:.2f} ms")
    print()
    print("model performance parameters")
    data={
      "Material":list(self.class_map.values()),
      "precision":precision_per_class,
      "recall":recall_per_class
    }
    df= pd.DataFrame(data)
    display(df)

In [ ]:
evaluator = model_performace_metrics(
    model=model,
    model_file_path='best_performing_model.keras',
    class_map= class_map
)

In [ ]:
evaluator.scorecard(X_test, y_test)